# Интеллектуальный парсинг сообщений 112 через локальную LLM Ollama и геокодирование по справочнику `settlments.csv`
Этот ноутбук развивает базовый алгоритм научного руководителя: вместо прямого поиска подстроки он отправляет сообщение в локальную LLM **Ollama**, получает название населённого пункта в **именительном падеже** и ориентир направления, после чего находит координаты в `settlments.csv`.

In [ ]:
from geocoder import SettlementGeocoder
from llm_client import hybrid_analyze_message

# 1. Загружаем справочник населённых пунктов Республики Башкортостан
geocoder = SettlementGeocoder("settlments.csv")
print(f"Загружено НП: {len(geocoder.records)}, районов: {len(geocoder.districts)}")

In [ ]:
# 2. Тестовые сообщения (включая Шаран и одноимённые НП в разных районах)
sample_messages = [
    "14:10, звонок в 112: житель села Шаран сообщил, что низко над посёлком пролетел беспилотник в сторону Туймазов.",
    "14:25, заметили беспилотник над селом Михайловка Уфимского района, движется в сторону Уфы.",
    "14:40, житель деревни Михайловка Абзелиловского района видел БПЛА, полетел на Белорецк.",
    "15:05, очевидцы сообщают, что над деревней Алексеевка заметили воздушный объект, летит на северо-восток."
]

for msg in sample_messages:
    extracted, engine = hybrid_analyze_message(msg, geocoder)
    geo = geocoder.resolve_observation(extracted)
    print("Сообщение:", msg)
    print("Контур:", engine, "| Извлечено:", extracted)
    if geo["found"]:
        s = geo["settlement"]
        print(f"Найдено: {s['full_name']} | {s['district']} -> ({s['lat']}, {s['lon']}) | Одноимённых в РБ: {geo['total_homonyms']}")
        if geo["vector"]:
            print("Вектор:", geo["vector"]["description"])
    print("-" * 75)